In [ ]:
import sys, os
sys.path.append("..")

from dotenv import load_dotenv
load_dotenv("../.env", override=True)

import os
print("ENV            :", os.getenv("ENV"))
print("LLM_MODEL      :", os.getenv("LLM_MODEL"))
print("REDIS_URL      :", os.getenv("REDIS_URL"))
print("VECTOR DIR     :", os.getenv("LOCAL_VECTOR_DIR"))
print("PRODUCTS_SOURCE:", os.getenv("PRODUCTS_SOURCE"))

from app.settings import settings
print("Settings loaded for env:", settings.env)


In [ ]:
from app.rag.loaders_production import load_documents
from app.rag.retriever_production import RAGRetriever

print("=" * 60)
print("Adding New Document to RAG Index")
print("=" * 60)

# Initialize retriever and load existing index
retriever = RAGRetriever()
retriever.load_index()

# Get current state
old_metadata = retriever.get_metadata()
print(f"📊 Current index state:")
print(f"   Version: {old_metadata.version}")
print(f"   Documents: {old_metadata.num_documents}")
print(f"   Chunks: {old_metadata.num_chunks}")

# Load only the NEW document(s)
# Assuming you added a new file: data/raw/new_guideline.txt
print(f"\n📄 Loading new documents from data/raw...")
new_docs = load_documents("data/raw")  # Loads all documents including new one

# Incremental update - FAST (seconds instead of minutes)
print(f"🔨 Adding new content to index (incremental)...")
new_metadata = retriever.build_index(new_docs, save=True, incremental=False)

# Show what changed
print(f"\n✅ Update complete!")
print(f"   Old version: {old_metadata.version}")
print(f"   New version: {new_metadata.version}")
print(f"   Documents: {old_metadata.num_documents} → {new_metadata.num_documents}")
print(f"   Chunks: {old_metadata.num_chunks} → {new_metadata.num_chunks}")
print(f"   Added: {new_metadata.num_chunks - old_metadata.num_chunks} new chunks")

print("=" * 60)

Version: 20251218_120301
Created: 2025-12-18T12:03:01.461824
Chunks: 5


In [5]:
from app.rag.loaders import load_documents
from app.rag.retriever import RAGRetriever

retriever = RAGRetriever()

# Try to load existing index
loaded = retriever.load_index()

if loaded:
    # Index exists but check if metadata exists
    metadata = retriever.get_metadata()
    
    if metadata is None:
        print("⚠️  Old index found without metadata")
        print("🔄 Rebuilding to create metadata...")
        
        # Load documents and rebuild
        docs = load_documents("data/raw")
        print(f"📄 Found {len(docs)} documents")
        
        # Force rebuild to create metadata
        metadata = retriever.build_index(docs, save=True)
        print("✅ Index rebuilt with metadata")
        print(f"   Version: {metadata.version}")
        print(f"   Chunks: {metadata.num_chunks}")
    else:
        print("✅ Loaded index with metadata")
        print(f"   Version: {metadata.version}")
        print(f"   Created: {metadata.created_at}")
        print(f"   Chunks: {metadata.num_chunks}")
else:
    # No index exists, build new one
    print("📁 No existing index, building new one...")
    docs = load_documents("data/raw")
    print(f"📄 Found {len(docs)} documents")
    
    metadata = retriever.build_index(docs, save=True)
    print("✅ Index built with metadata")
    print(f"   Version: {metadata.version}")
    print(f"   Chunks: {metadata.num_chunks}")

✅ Loaded index with metadata
   Version: 20251218_120301
   Created: 2025-12-18T12:03:01.461824
   Chunks: 5


In [ ]:
import redis
from app.settings import settings

r = redis.from_url(settings.redis_url, decode_responses=True)
print("PING →", r.ping())


In [1]:
from pprint import pprint

def show(resp):
    print("session_id     :", resp.get("session_id"))
    print("intent         :", resp.get("intent"))
    print("used_rag       :", resp.get("used_rag"))
    print("rag_best_score :", resp.get("rag_best_score"))
    print("redaction      :", resp.get("redaction"))
    print("scope          :", resp.get("scope"))
    print("escalation?    :", "YES" if resp.get("escalation_banner") else "no")
    print("\n--- ANSWER ---\n")
    print(resp.get("answer", "")[:1200])
    print("\n--- PRODUCTS ---")
    for p in resp.get("products", []):
        print("-", p.get("name"))
    print("\n--- MEMORY (recent) ---")
    for t in resp.get("memory_turns", []):
        print(f"{t['role'].upper()}: {t['content'][:160]}")


In [ ]:
from app.chains.chat_pipeline import chat_once

# res1 = chat_once(
#     "Who is rahul dravid?",
#     session_id=None
# )

# res1 = chat_once(
#     "What are some tips to recover faster after childbirth?",
#     session_id=None
# )

res1 = await chat_once(
    "Which breast pump is better for daily use as a new mother?",
    session_id=None
)

print("\n\n=== CHAT RESPONSE ===\n",res1)

show(res1)

sid = res1["session_id"]
sid



In [ ]:
# res1 = chat_once(
#     "My baby has a high fever. Should I give paracetamol?",
#     session_id=None
# )
# show(res1)


In [ ]:
# res1 = chat_once(
#     "Can you suggest a breast pump brand that is not in your product list?",
#     session_id=None
# )
# show(res1)

In [ ]:
res1 = chat_once(
    "My name is Riya, but don’t use it. Just answer normally,What are some tips to recover faster after childbirth?",
    session_id=None
)